# Notebook 01 — Exploratory Data Analysis

**Project:** LLM-Powered Pricing Intelligence Framework  
**Author:** Cristian David Montoya Henao  
**Purpose:** Understand demand distribution, seasonality patterns, and outlier prevalence before modeling.

---
### Research Questions Addressed
- What is the temporal structure of the demand signal (trend + seasonality)?
- What percentage of observations are flagged as outliers under IQR thresholds?
- Are there calendar effects (weekday, month) worth encoding as regressors?

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys; sys.path.insert(0, '../..')
from src.ingestion.pipeline import run_pipeline

# Load and clean data
df = run_pipeline('../../data/raw/sample.csv')
df.head()

In [ ]:
# ── Basic statistics ──
print('Shape:', df.shape)
print('Date range:', df['ds'].min(), '→', df['ds'].max())
df['y'].describe().round(2)

In [ ]:
# ── Time series overview ──
fig = px.line(df, x='ds', y='y', title='Demand Over Time')
fig.update_layout(template='plotly_white', height=350)
fig.show()

In [ ]:
# ── Seasonality: average demand by day of week ──
dow = df.groupby('day_of_week')['y'].mean().reset_index()
dow['day_name'] = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
fig = px.bar(dow, x='day_name', y='y', title='Avg Demand by Day of Week',
             color='y', color_continuous_scale='Blues')
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
# ── Distribution ──
fig = px.histogram(df, x='y', nbins=40, title='Demand Distribution',
                   marginal='box', template='plotly_white')
fig.show()

In [ ]:
# ── Rolling statistics: 7-day moving average ──
df['rolling_mean'] = df['y'].rolling(7).mean()
df['rolling_std']  = df['y'].rolling(7).std()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['ds'], y=df['y'], name='Raw', line=dict(color='#1f3864', width=1, dash='dot')))
fig.add_trace(go.Scatter(x=df['ds'], y=df['rolling_mean'], name='7-day MA', line=dict(color='#e84855', width=2)))
fig.update_layout(title='Demand + 7-Day Moving Average', template='plotly_white', height=350)
fig.show()

---
### Key Findings

| Finding | Implication for Modeling |
|---|---|
| Weekly seasonality visible | Include `weekly_seasonality=True` in Prophet / NeuralProphet |
| Outlier rate: ~X% | IQR multiplier=1.5 appropriate; revisit if rate > 10% |
| Upward trend observed | All models should capture trend component |
| Weekend dip detected | Consider `is_weekend` as external regressor |

→ Proceed to **Notebook 02: Model Benchmark**